# Stage 2 Notebook 65 - Exp2JJJ Anchor + cls_sep + topk_fixed K=3 + full 70K + 12ep

**Stable matching on top of NB62's recipe.** NB55 showed K=3 topk_fixed gives matched_iou=0.381 at limit=3000 (recovered from K=8's collapse). The stable labels (no batch flicker) should help cls separation when combined with the cls_sep feature pathway and full data.

Single diff vs NB62 (exp57): `lane_assigner: dynamic_k -> topk_fixed`, `topk_fixed_per_gt: 3`. All else identical (width=0.5, cls_sep=true, full 70K, 12 epochs, bb_throttle=0.01).

Tests whether stable per-prior labels combine constructively with cls_sep to give more cls discrimination than either alone.

### Run mode
1. Smoke.
2. 12 epochs full 70K. ~2.5-3 hr.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp60_rmt_gca_anchor_cls_sep_topk3_vfl_full_data_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp60_rmt_gca_anchor_cls_sep_topk3_vfl_full_data_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp60_rmt_gca_anchor_cls_sep_topk3_vfl_full_data_joint_smoke.log
OK exp60_rmt_gca_anchor_cls_sep_topk3_vfl_full_data_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=5.0108 det_loss=3.1761 grad_cos=-0.0327 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.5019611716270447, 'gate/lane_mean': 0.4995841681957245, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp60_rmt_gca_anchor_cls_sep_topk3_vfl_full_data_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full12'
    EPOCHS = 12
    BATCH_SIZE = 8
    LIMIT_TRAIN = None
    LIMIT_VAL = 2000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: None
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp60_rmt_gca_anchor_cls_sep_topk3_vfl_full_data_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp60_rmt_gca_anchor_cls_sep_topk3_vfl_full_data_joint_full12 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp60_rmt_gca_anchor_cls_sep_topk3_vfl_full_data_joint_full12.tar --epochs 12 --batch-size 8 --limit-val 2000 --force-extract --print-every 50
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp60_rmt_gca_anchor_cls_sep_topk3_vfl_full_data_joint_full12.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp60_rmt_gca_anchor_cls_sep_topk3_vfl_full_data_joint_full12_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp60_rmt_gca_

0

## What to watch in Exp2JJJ

Reference NB62 (dynamic_k + cls_sep + full 12ep): matched_iou=0.550, decoded_f1=0.073, gap=0.045, val_lane_f1=0.118.
Reference NB55 (topk_fixed K=3, 3K, NO cls_sep): matched_iou=0.381, gap=0.002.

Pass criteria at epoch 12:
- val/matched_line_iou >= 0.45 (some geometry loss vs NB62 expected from K=3 dilution).
- **pos-neg gap >= 0.08** (2x NB62; topk_fixed's stable labels + cls_sep's disjoint parameters should multiply).
- val/lane/decoded_f1 >= 0.08.
- val/lane_f1 >= 0.20.